In [2]:
# pandas → 이온주입/어닐링 데이터를 표 형태로 관리
# numpy → 실습용 가상 데이터 생성 및 로그 변환
# LinearRegression → 공정 파라미터와 면저항의 관계 분석

import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression

In [3]:
# 실제 implant_anneal_data.csv가 없으므로 실습용 가상 데이터 생성
# 실행할 때마다 동일한 결과가 나오도록 난수 고정

np.random.seed(42)

# 실습 데이터 개수
n = 150

# 이온주입 Dose
# 실제 공정에서는 매우 큰 수를 사용하므로 10^13 ~ 10^15 범위로 가정
dose = 10 ** np.random.uniform(13, 15, n)

# 이온주입 Energy (keV)
energy_kev = np.random.uniform(
    10,
    100,
    n
)

# 어닐링 온도 (°C)
anneal_temp = np.random.uniform(
    800,
    1050,
    n
)

# 어닐링 시간 (sec)
anneal_time = np.random.uniform(
    5,
    60,
    n
)

In [4]:
# Sheet Resistance 생성
# 학습용으로 아래 관계를 가정
#
# Dose 증가        → 면저항 감소
# Energy 증가      → 면저항 일부 변화
# Anneal Temp 증가 → 도펀트 활성화 증가 → 면저항 감소
# Anneal Time 증가 → 면저항 감소 경향
#
# 실제 공정 관계는 공정 조건에 따라 달라질 수 있음

dose_log = np.log10(dose)

sheet_resistance = (
    800
    - 70 * dose_log
    + 1.2 * energy_kev
    - 0.35 * anneal_temp
    - 0.8 * anneal_time
    + np.random.normal(0, 20, n)
)

In [5]:
# 분석에 사용할 데이터프레임 생성

df = pd.DataFrame({
    'DOSE': dose,
    'ENERGY_KEV': energy_kev,
    'ANNEAL_TEMP': anneal_temp,
    'ANNEAL_TIME': anneal_time,
    'SHEET_RESISTANCE': sheet_resistance
})

df.head()

,DOSE,ENERGY_KEV,ANNEAL_TEMP,ANNEAL_TIME,SHEET_RESISTANCE
0,5.611516e+13,91.743930,812.920430,47.743080,-385.664668
1,7.969455e+14,31.560570,932.838658,35.712234,-576.148093
2,2.910636e+14,23.040538,935.158780,28.332211,-536.942454
3,1.575132e+14,54.050748,959.357475,54.849491,-529.310845
4,2.051338e+13,98.708541,981.522833,11.115862,-376.891028


In [6]:
# 데이터 크기와 컬럼 확인

print("데이터 크기:", df.shape)
print("컬럼:")
print(df.columns.tolist())

df.describe()

데이터 크기: (150, 5)
컬럼:
['DOSE', 'ENERGY_KEV', 'ANNEAL_TEMP', 'ANNEAL_TIME', 'SHEET_RESISTANCE']


,DOSE,ENERGY_KEV,ANNEAL_TEMP,ANNEAL_TIME,SHEET_RESISTANCE
count,1.500000e+02,150.000000,150.000000,150.000000,150.000000
mean,2.022722e+14,56.576986,924.038120,33.903079,-460.586890
std,2.473729e+14,26.227062,75.334606,16.727386,62.822929
min,1.025756e+13,10.455543,802.709413,5.624450,-626.969559
25%,2.689339e+13,32.272988,862.089742,19.411928,-509.747982
50%,7.879780e+13,60.040093,926.048579,34.254431,-461.575349
75%,3.160779e+14,78.205381,988.281818,49.677275,-412.426291
max,9.413993e+14,99.104847,1047.626286,59.984472,-323.922405


In [7]:
# Dose 값은 10^13 ~ 10^15처럼 자릿수 차이가 크므로
# 로그 변환하여 분석하기 쉽게 만듦

df['DOSE_LOG'] = np.log10(
    df['DOSE']
)

df[['DOSE', 'DOSE_LOG']].head()

,DOSE,DOSE_LOG
0,5.611516e+13,13.749080
1,7.969455e+14,14.901429
2,2.910636e+14,14.463988
3,1.575132e+14,14.197317
4,2.051338e+13,13.312037


In [8]:
# 입력 변수
# DOSE_LOG      → 이온주입량
# ENERGY_KEV    → 이온 에너지
# ANNEAL_TEMP   → 어닐링 온도
# ANNEAL_TIME   → 어닐링 시간

features = [
    'DOSE_LOG',
    'ENERGY_KEV',
    'ANNEAL_TEMP',
    'ANNEAL_TIME'
]

X = df[features].values

# 예측 대상
# Sheet Resistance
y = df['SHEET_RESISTANCE'].values

In [9]:
# 선형회귀 모델 생성

model = LinearRegression()

# 공정 파라미터와 Sheet Resistance 관계 학습
model.fit(X, y)

LinearRegression()

In [10]:
# 각 변수의 회귀계수 확인
#
# 계수 > 0
# → 해당 변수가 증가할수록 Sheet Resistance 증가 경향
#
# 계수 < 0
# → 해당 변수가 증가할수록 Sheet Resistance 감소 경향

print("회귀 계수:")

for feat, coef in zip(
    features,
    model.coef_
):
    print(
        f"{feat:20s}: {coef:+.4f}"
    )

회귀 계수:
DOSE_LOG            : -69.8044
ENERGY_KEV          : +1.2824
ANNEAL_TEMP         : -0.3628
ANNEAL_TIME         : -0.9780


In [11]:
# R²는 모델이 Sheet Resistance 변동을
# 얼마나 설명하는지 나타내는 지표
#
# 1에 가까울수록 현재 데이터에 대한 설명력이 높음

r2 = model.score(
    X,
    y
)

print(f"R² score: {r2:.4f}")

R² score: 0.9070


In [12]:
# 학습된 모델로 Sheet Resistance 예측

df['PREDICTED_RS'] = model.predict(X)

df[
    [
        'DOSE',
        'ENERGY_KEV',
        'ANNEAL_TEMP',
        'ANNEAL_TIME',
        'SHEET_RESISTANCE',
        'PREDICTED_RS'
    ]
].head(10)

,DOSE,ENERGY_KEV,ANNEAL_TEMP,ANNEAL_TIME,SHEET_RESISTANCE,PREDICTED_RS
0,5.611516e+13,91.743930,812.920430,47.743080,-385.664668,-374.977795
1,7.969455e+14,31.560570,932.838658,35.712234,-576.148093,-564.340427
2,2.910636e+14,23.040538,935.158780,28.332211,-536.942454,-538.355527
3,1.575132e+14,54.050748,959.357475,54.849491,-529.310845,-514.686079
4,2.051338e+13,98.708541,981.522833,11.115862,-376.891028,-360.887351
5,2.051110e+13,31.784974,1043.963020,32.094381,-508.717383,-489.881933
6,1.306674e+13,70.492199,929.075087,5.624450,-323.922405,-359.001206
7,5.399484e+14,78.545765,880.739118,30.776335,-469.184231,-468.552085
8,1.593052e+14,31.387379,998.796549,8.096680,-526.541893,-512.677001
9,2.607025e+14,75.539471,867.708063,11.534985,-427.129007,-426.788350
